In [1]:
import torch
import nltk

from torch import nn
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    EncoderDecoderModel,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    AutoModel
)


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

torch.set_default_device(device)
nltk.download('punkt')

Using device: cuda


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
encoder_base, decoder_base = "DeepPavlov/rubert-base-cased", "DeepPavlov/rubert-base-cased"

tokenizer = BertTokenizer.from_pretrained(encoder_base)

# Генеративный BERT с hf лучше работает с cls и sep токенами как bos и eos
tokenizer.bos_token = tokenizer.cls_token   # [CLS]
tokenizer.eos_token = tokenizer.sep_token   # [SEP]

### Data

In [3]:
dataset = load_dataset("IlyaGusev/gazeta")

train_data = dataset["train"].select(range(40000))
val_data = dataset["validation"].select(range(6369))

In [37]:
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    # токенизируем входную статью
    encoder_inputs = tokenizer(examples["text"], max_length=max_input_length, truncation=True)
    # токенизируем целевой суммари
    labels = tokenizer(examples["summary"],
                        max_length=max_target_length,
                        truncation=True)

    encoder_inputs["labels"] = labels["input_ids"]
    return encoder_inputs

tokenized_train = train_data.map(preprocess_function, batched=True,
                                 remove_columns=train_data.column_names)
tokenized_val = val_data.map(preprocess_function, batched=True,
                             remove_columns=val_data.column_names)

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

In [48]:
tokenized_train

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 40000
})

### Model

In [5]:
model = EncoderDecoderModel.from_encoder_decoder_pretrained(
    encoder_base, decoder_base
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertLMHeadModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                                                | Status     | 
-------------------------------------------------------------------+------------+-
bert.embeddings.position_ids                                       | UNEXPECTED | 
cls.seq_relationship.bias                                          | UNEXPECTED | 
bert.pooler.dense.bias               

In [12]:
from transformers import GenerationConfig


generation_config = GenerationConfig(
    max_length=128,
    no_repeat_ngram_size=3,
    length_penalty = 1.0,
    eos_token_id=tokenizer.sep_token_id,
    decoder_start_token_id=tokenizer.cls_token_id,
    pad_token_id=tokenizer.pad_token_id,
)

model.generation_config = generation_config
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id = tokenizer.sep_token_id
model.config.pad_token_id = tokenizer.pad_token_id

### Evaluation Metrics

In [31]:
!pip install -q evaluate rouge_score
import evaluate


rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # Decode generated summaries into text
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in the labels as we can't decode them.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # Decode reference summaries into text
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Rouge expects a newline after each sentence
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

  Preparing metadata (setup.py) ... done


In [46]:
batch_size = 8
training_args = Seq2SeqTrainingArguments(
    output_dir="./bert2bert",
    save_strategy="epoch",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    logging_steps=100,
    predict_with_generate=True,
    fp16=True,
    dataloader_pin_memory=False,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [49]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

trainer.train()

Step,Training Loss
100,3.911240
200,4.841791
300,4.688949
400,4.631620
500,4.555676
600,4.481720
700,4.477876
800,4.400388
900,4.396124
1000,4.359598


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15000, training_loss=3.182812214152018, metrics={'train_runtime': 7194.0776, 'train_samples_per_second': 16.68, 'train_steps_per_second': 2.085, 'total_flos': 1.0755849850734182e+17, 'train_loss': 3.182812214152018, 'epoch': 3.0})

In [53]:
def generate_summary(text, model, tokenizer, max_length=128):
    inputs = tokenizer(text, max_length=512, truncation=True,
                      return_tensors="pt").to(model.device)
    output_ids = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_length=max_length,
        num_beams=4,
        no_repeat_ngram_size=3,
        length_penalty=2.0,
        early_stopping=True
    )
    summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return summary


test_article = """
Производитель пылесосов хочет обогнать Apple

Dreame Technology занимается разработкой и производством автономных пылесосов, однако её гендиректор Ю Хао решил бодаться с Apple на их поле. Он опубликовал в Weibo пост, где сообщил что «Dreame может превзойти Apple, потому что Apple перестала внедрять инновации», а также добавил, что «Dreame унаследует наследие Стива Джобса, победит Apple и превзойдет Apple».

Также Ю стал собирать отзывы о недостатках iPhone и продвигать инновационные идеи от представителей индустрии мобильных телефонов. Парой дней ранее Dreame показала модульный смартфон Aurora Nex LS1, который может превращаться в фотоаппарат, экшн-камеру или спутниковый телефон. При этом ни даты выхода, ни цены пока нет, а утверждения Ю выглядят обычным китайским самопиаром.
"""

print(generate_summary(test_article, model, tokenizer))

Производитель пылесосов для Apple заявил, что хочет превзойти Apple по популярности гаджетов. По словам главы компании Ю Хаоа, гаджеты Apple могут быть полезны для гаджетов, а не только для того, чтобы конкурировать с Apple и Apple.


In [51]:
save_path = "./bert2bert/result/"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bert2bert/result/tokenizer_config.json',
 './bert2bert/result/tokenizer.json')